# Phase 6.5 shard 13 (forest65)

Runs **108 cells** of the frozen Phase 6.5 manifest (`G3-PHASE65-v1`), covering: `causal_drf`, `causal_drf_log`, `causal_drf_retn`, `drf`, `drf_log`.

This shard runs the R forest baselines, including the two adversarial controls (log geometry and bandwidth retune). The setup cell installs R, the pinned `drf` 1.3.1, and the authors' causal-clean package at the frozen commit; fifteen to twenty-five minutes.

Estimated single-threaded compute on the reference machine is about **87 minutes**. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip`; collect every shard's zip into `results/phase65/colab_shards/` (logs into `results/manifests/`) and run `python research/run_phase65.py merge`.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported.
# OpenMP sizes its pool at initialisation, so setting these
# afterwards is silently ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Clone the repository at the pinned commit

Remote `https://github.com/hugogobato/wasserstein-causal-forests.git`, commit `5f3b2ad66b0b`. After checkout the notebook asserts the frozen manifest checksum, so a clone of anything but the generating commit fails here rather than mid-run.

In [ ]:
import subprocess, pathlib, os, sys, json, hashlib

REPO = 'https://github.com/hugogobato/wasserstein-causal-forests.git'
COMMIT = '5f3b2ad66b0b05c8f8fed19e0d849d75a206cbe8'
EXPECTED_CHECKSUM = '4e28d308ca99cde4c81379524fc4492a15b38f029b449899b0a307b6c0ace110'

workdir = pathlib.Path('/content/wcf')
if not workdir.exists():
    subprocess.run(['git', 'init', '-q', str(workdir)], check=True)
    subprocess.run(
        ['git', '-C', str(workdir), 'remote', 'add', 'origin', REPO],
        check=True,
    )
# A shallow fetch of the exact commit: nothing else is downloaded.
    subprocess.run(
        ['git', '-C', str(workdir), 'fetch', '-q', '--depth', '1',
         'origin', COMMIT], check=True,
    )
    subprocess.run(
        ['git', '-C', str(workdir), 'checkout', '-q', 'FETCH_HEAD'],
        check=True,
    )
os.chdir(workdir)
sys.path.insert(0, str(workdir / 'src'))
os.environ['WCF_CAUSAL_DRF_R_LIB'] = '/content/wcf/results/Rlib/causal_drf'

manifest = json.load(open(
    'results/manifests/phase65_manifest.json', encoding='utf-8'
))
checksum = hashlib.sha256(
    json.dumps(manifest['cells'], sort_keys=True).encode('utf-8')
).hexdigest()
assert checksum == EXPECTED_CHECKSUM, (
    'the cloned manifest does not match the frozen grid: '
    f'{checksum} != {EXPECTED_CHECKSUM}'
)
print('repo ready at commit ' + COMMIT[:12] + '; '
      + str(manifest['n_cells']) + ' frozen cells verified')

## 2. Dependencies

In [ ]:
# This group runs the R forest baselines, including Causal-DRF
# through the authors' causal-clean package at the frozen commit.
# Expect fifteen to twenty-five minutes for this cell.
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite","remotes","transport"), repos="https://cloud.r-project.org", quiet=TRUE)'
Rscript -e 'options(Ncpus=2); install.packages("https://cran.r-project.org/src/contrib/Archive/drf/drf_1.3.1.tar.gz", repos=NULL, type="source", quiet=TRUE)' || Rscript -e 'options(Ncpus=2); install.packages("drf", repos="https://cloud.r-project.org", quiet=TRUE)'
mkdir -p results/Rlib/causal_drf
Rscript -e 'options(Ncpus=2); .libPaths(c("results/Rlib/causal_drf",.libPaths())); remotes::install_github("herbps10/drf", ref="0a1a508444176b5b1553f13e832be93a374b0af2", lib="results/Rlib/causal_drf", upgrade="never", quiet=TRUE)'
Rscript -e 'cat("drf", as.character(packageVersion("drf")), "ready\n")'
echo 'setup complete'


## 3. This shard's cells

In [ ]:
import json, collections
SHARD_INDEX = 13
CELLS = json.loads('''[{"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "de7dc4e318bc1c2a", "test_seed": 900001}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "9fb4ca529744e723", "test_seed": 900001}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "5abc923d92534ed9", "test_seed": 900001}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "a253d6474cba8467", "test_seed": 900001}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "4cdfe9a317040751", "test_seed": 900001}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "f63fe9ce62c2c80c", "test_seed": 900001}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "6a54439482647dc1", "test_seed": 900001}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "e66293b4b9a18ae0", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "df3b32104d706c88", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "82d49c6518c17f2b", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "1fb3400a3cc6394c", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "e277d998f9d95902", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "d64587089f70b5b8", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "615506782cba3e20", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "f72665d2c8de82a3", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "3dfce92c15014673", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "aff2823081e39fbf", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "0af27ef99181cb5f", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "255c3336791b3c23", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "7abbcc0f958e9500", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "645b0b342ad4ae6a", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "e8b2e24c74b996c7", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 1, "cell_key": "36bb45dfbb9e1a6c", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 6, "cell_key": "b58f9ca0da657dec", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "f204986cd1950466", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "6fa6ae7a2fc785a0", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "3b7278396ffffd4b", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "4398150926b56c39", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "5eac7403c36915ef", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "a87e65d4ef2ccea7", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "04e9cb572d6b81d3", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "4783a997b89772a8", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "6bea240088274b1c", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "73a876f5e0c0380b", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "0b7f95bbf6a2d65d", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "10f158fa20de8e35", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "b7383973548ba0e5", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "92c48db856c9b9ab", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "8744d3b03a68eb56", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "cffee584045207a7", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "b2e26fefcc3d9e88", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "0f930fb8e35934a3", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "9395bea5e5b62b55", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "eee0b671f32dab2f", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "0c312fbfb501c5a9", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "e52d8f57c081d2c3", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "e53ae45bf98b7865", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "f1c608047c1d9ee8", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "8de17947700a64c2", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "9a570083e0c49862", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "8c9e82416e5563ed", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "fb187c74058a75d3", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "605319364d528827", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "857c451aad6bedfe", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "8168d787ae830996", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "3fae1248e7f852a9", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "b1986b6edb1ae58d", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "aec4063182257d9e", "test_seed": 900006}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "e36dd1280521f540", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "23446acd05db1fd5", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "c51b3be77ea0317b", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "24199754a8f53656", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "531887e82c505708", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "7e7b8f96e7ebfbfc", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "882555bfd22e96c3", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "74a748a6467ab1f0", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "2a6a393a84e7eb62", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "796a9013cd1b9438", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "e3fa8698cdedf8f9", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "52319b2e2dbe0a55", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "b483ed20eba65e3c", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "aa63604d7a344c46", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "2577688ffda22df3", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "bc7a51fc8fd70875", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "bbae0019487baa09", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "0ccf431c72a6660a", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "664a4d28fb8177d1", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "895cb0024707092e", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "62458d5119913936", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "cf80b5f38d534d49", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "cd61df0931033f40", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "f8fb0621eda8fac2", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "c777f9b8c8228249", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "f164d9d7323f56a1", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "8073999c017d023a", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "27b9bb3c4eb9a93c", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "9f9559bf5651ded8", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "75c19c88acfa3a06", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 1, "cell_key": "2a5338463e415c36", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 6, "cell_key": "22645c4300a4b4e1", "test_seed": 900006}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 1, "cell_key": "9dae13e9dc7d04f9", "test_seed": 900001}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 6, "cell_key": "2c129eeff7b3e11b", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "040b5a9fcef6b98e", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "450d6afc87bc059e", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "dbdc5ae959528761", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "23b6d530323fe5e6", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "45c142f3e151e728", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "94cf52057d9bfe8b", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "8759c253c2a7a193", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "c058f9457fedc04e", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "41daff3de9bdc694", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "fbffc8ace772a2ff", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "c4c2a293368522df", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "fb3c5394bd6817d0", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "b3d02b982f2b6be7", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "9ff08ff439041695", "test_seed": 900006}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "5ee3d15c61069db2", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 6, "cell_key": "9f61afb38137f433", "test_seed": 900006}]''')
print(f'{len(CELLS)} cells in this shard')
for key, count in sorted(collections.Counter(
        (c['grid'], c['dgp'], c['method'])
        for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:8s} {key[2]:18s} {count}')

## 4. Bandwidth-selection pilot (preregistered)

This shard contains `causal_drf_retn` cells, so it first runs the selection pilot on seeds 100 and 101, outside every decisive range, and freezes the multipliers document. The rule picks the candidate with the best held-out energy score; oracle truth is never read.

In [ ]:
from pathlib import Path
import json, numpy as np
from wasserstein_causal_forests.g3.dgps import build_dgp
from wasserstein_causal_forests.g3.phase65_methods import (
    BANDWIDTH_CANDIDATES, SELECTION_SEEDS, select_bandwidth_multiplier,
)

keys = sorted({(c['dgp'], c['n_train']) for c in CELLS
               if c['method'] == 'causal_drf_retn'})
multipliers = {}
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)
for dgp_name, n_train in keys:
    dgp = build_dgp(dgp_name, 25)
    best, means = select_bandwidth_multiplier(
        dgp, n_train, seeds=SELECTION_SEEDS,
        candidates=BANDWIDTH_CANDIDATES, cache_directory=cache,
    )
    multipliers[f'{dgp_name}|{n_train}'] = best
    scores = {str(k): round(v, 5) for k, v in means.items()}
    print(f'{dgp_name} n={n_train}: multiplier {best}  scores {scores}',
          flush=True)

document = {
    'rule': 'held-out energy score, pilot seeds 100 and 101, '
            'candidates ' + repr(BANDWIDTH_CANDIDATES),
    'multipliers': multipliers,
}
path = Path('/content/wcf/results/manifests/'
            'phase65_bandwidth_selection.json')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(document, indent=2))
print('froze', path)

## 5. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/phase65/colab_shards')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/phase65_execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
    manifest_contract_id='G3-PHASE65-v1',
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept and reported at merge time; a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## Download the results

In [ ]:
import shutil
bundle = '/content/p65_shard_13_forest65'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
if log.exists():
    shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)